In [1]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-sonnet-4-6"

In [2]:
# Helper functions
from anthropic.types import Message


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[], tools=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

In [15]:
web_search_schema = {
    "type": "web_search_20250305",
    "name": "web_search",
    "max_uses": 5,
    "allowed_domains": ["nih.gov"],#National library of Medicine
}

## How the Response Works

When Claude uses the web search tool, the response contains several types of blocks:

- **Text blocks** - Claude's explanation of what it's doing
- **ServerToolUseBlock** - Shows the exact search query Claude used
- **WebSearchToolResultBlock** - Contains the search results
- **WebSearchResultBlock** - Individual search results with titles and URLs
- **Citation blocks** - Text that supports Claude's statements

In [16]:
messages = []
add_user_message(
    messages,
    """
    What's the best exercise for gaining leg muscle?
    """,
)

#Jerry Force to use web search tool
system="""
Always use the web_search tool before answering. Never answer from memory alone.
"""


response = chat(messages, system = system, tools=[web_search_schema])
response

Message(id='msg_015aah4XghiGKUvQopcrGdfu', container=None, content=[ServerToolUseBlock(id='srvtoolu_01TdW6iiAzmA98CqRpNEC9Mq', caller=None, input={'query': 'best exercises for gaining leg muscle'}, name='web_search', type='server_tool_use'), WebSearchToolResultBlock(caller=DirectCaller(type='direct'), content=[WebSearchResultBlock(encrypted_content='EpECCioIEBgCIiRlODhjOWM4Yi1jMzE4LTQxZGYtYWQ4ZC1mM2MxNGIxMmMxOWISDMNKdbEhDAtBrRoF2hoM8R9f+74GqU8D2VB9IjB4Z/igwjk6A3gK/pEQgF90i7PpNSHFdx4BO0z4/K923sHyaqcbdC4jaRq5ckXrIL8qlAG7F8ov82aSF/KtCSCw9ao9ffJMdnN3UgASVB8x78ShAHusYMEl8UG7Eq0WU85SsJeYwj/pi6r9gyCI78tRxMpAJI4HLA6u3yS8zXQ+ZnOVz2OoF6JbtROqKL4NoxxIneg1k0xcDvk1fQ+8WTZLyy8Ot9xb3u00bFsIbcdAmV5fpMfDhQ4bWgOMDHLzaQLuh8AalD/qGAM=', page_age=None, title='Maximizing Muscle Hypertrophy: A Systematic Review ... - PMC', type='web_search_result', url='https://pmc.ncbi.nlm.nih.gov/articles/PMC6950543/'), WebSearchResultBlock(encrypted_content='EqwgCioIEBgCIiRlODhjOWM4Yi1jMzE4LTQxZGYtYWQ4ZC1mM2MxNGIxMmMxOWIS